# Bird-detection training

Co-trains a few binary bird / no-bird detectors on shared raw-audio batches. Each writes its own `model.keras`, `train.json`, `model.tflite` and `results.json`. The split is materialized here from cached area clips, so run it once `21_detection_dataset_build` has populated the download cache.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os

import pyrootutils

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

## Parameters

In [ ]:
from building.polychirp_training import detection_slug
from building.models import BASE_LIST, TinyChirpModel, ds_cnn, mcunet_in0, mobilenetv2
from building.polychirp_training.training import TrainHP

LAT, LON = 48.8566, 2.3522  # Paris: which species belong to the zone

# Curated set plus detection-only extras (MobileNetV2/MCUNet-in0/DS-CNN baselines).
MODELS = BASE_LIST + [
    TinyChirpModel(base_model=ds_cnn.MODEL),
    TinyChirpModel(base_model=mobilenetv2.MODEL),
    TinyChirpModel(base_model=mcunet_in0.MODEL),
]
EXCLUDE = {"cnn1d", "transformer"}  # raw-time models, too slow to co-train here

EPOCHS = 100
PATIENCE = 10
MIN_DELTA = 1e-3
BATCH_SIZE = 32
SEED = 42

LR_PATIENCE = 5
LR_FACTOR = 0.5
LR_MIN = 1e-5

DETECTION_COLLECTION = detection_slug(LAT, LON)
all_specs = [s for s in MODELS if s.name not in EXCLUDE]
specs = [s for s in all_specs if not s.paths(DETECTION_COLLECTION).results_path.exists()]
HP = TrainHP(
    epochs=EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA,
    batch_size=BATCH_SIZE, seed=SEED,
    lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR, lr_min=LR_MIN,
    augment=True,
)
print(f"detection collection: {DETECTION_COLLECTION}")
print(f"training {len(specs)}/{len(all_specs)} models (skipping those already scored):")
for spec in specs:
    print(f"  {spec.name} x{spec.multi:g}  ->  {spec.paths(DETECTION_COLLECTION).dir}")


## Build & load detection dataset

In [ ]:
import logging
import warnings

import numpy as np
import tensorflow as tf

tf.get_logger().setLevel("ERROR")
logging.getLogger("absl").setLevel("ERROR")
warnings.filterwarnings("ignore", message=".*AudioResample.*")

from building.data.dataset import build_detection_dataset
from building.polychirp_training.training import (
    build_grouped_dataset_from_catalog,
    collect_predictions,
    load_dataset_catalog,
)

build_detection_dataset(DETECTION_COLLECTION)
catalog = load_dataset_catalog(DETECTION_COLLECTION)
bird_idx = catalog.class_names.index("bird")
no_bird_idx = catalog.class_names.index("no_bird")
print(f"classes: {catalog.class_names}  (bird={bird_idx}, no_bird={no_bird_idx})")

## Binary bird detectors (co-trained)

In [ ]:
groups = [[bird_idx], [no_bird_idx]]
label_names = ["bird", "no_bird"]
rng = np.random.default_rng(SEED)

train_ds, meta = build_grouped_dataset_from_catalog(
    catalog, groups, BATCH_SIZE, rng,
    split="train", input_repr=None, augment=True, balance="uniform",
)
val_ds, _ = build_grouped_dataset_from_catalog(
    catalog, groups, BATCH_SIZE, rng,
    split="val", input_repr=None, augment=False,
)
test_ds, _ = build_grouped_dataset_from_catalog(
    catalog, groups, BATCH_SIZE, rng,
    split="test", input_repr=None, augment=False,
)
print(f"epoch_samples={meta.epoch_samples:,}  "
      f"bird train={catalog.entries[bird_idx].train.count:,}  "
      f"no_bird train={catalog.entries[no_bird_idx].train.count:,}")

In [ ]:
import gc
from building.polychirp_training.training import SpecState

tf.keras.backend.clear_session()
gc.collect()
states = [SpecState(spec=s, model=s.builder(meta.n_classes)) for s in specs]
for st in states:
    print(f"{st.spec.name} x{st.spec.multi:g}  params={st.model.count_params():,}")

In [ ]:
from building.polychirp_training.training import co_train
from building.polychirp_training.results import MultiModelLivePlot, spec_label

live = MultiModelLivePlot([spec_label(s) for s in specs], legend=True)


def restore_best(state):
    if state.best_weights is not None:
        state.model.set_weights(state.best_weights)


try:
    co_train(
        states, train_ds, val_ds, hp=HP,
        on_epoch_end=lambda st, _e, logs: live.update(spec_label(st.spec), logs),
        on_spec_stop=restore_best,
    )
except KeyboardInterrupt:
    print("\ninterrupted, restoring best weights")
    for st in states:
        restore_best(st)

for st in states:
    print(f"{spec_label(st.spec)}: {st.epochs_trained} epochs  best val_loss={st.best_val_loss:.4f}")


## Quantize, score, and save

Bake each detector to INT8, evaluate bird detection on the test split at F1- and F2-optimal thresholds (tuned on validation), and write `train.json` / `results.json` per detector.

In [ ]:
from building.models.bake import analyze_tflite, bake_model
from building.models import model_eval as M
from building.polychirp_training.training import featurize_for_eval

stats: dict[str, M.TFLiteStats] = {}
for st in states:
    label = spec_label(st.spec)
    paths = st.spec.paths(DETECTION_COLLECTION)
    paths.dir.mkdir(parents=True, exist_ok=True)
    st.model.save(paths.model_path)
    if paths.tflite_path.exists():
        stats[label] = analyze_tflite(paths.tflite_path)
        print(f"[{label}] INT8 reused: {paths.tflite_path}")
    else:
        rep = featurize_for_eval(val_ds, st.spec.input_frontend)
        stats[label] = bake_model(st.model, rep, paths.tflite_path, n_representative=100)
        print(f"[{label}] INT8 -> {paths.tflite_path}")


In [ ]:
from building.polychirp_training import results as R
from building.polychirp_training.training import TrainSidecar

bird = label_names.index("bird")
bce = tf.keras.losses.BinaryCrossentropy()

for st in states:
    label = spec_label(st.spec)
    paths = st.spec.paths(DETECTION_COLLECTION)
    val_feat = featurize_for_eval(val_ds, st.spec.input_frontend)
    test_feat = featurize_for_eval(test_ds, st.spec.input_frontend)
    y_val, s_val = collect_predictions(st.model, val_feat)
    y_test, s_test = collect_predictions(st.model, test_feat)
    _, q_val, _ = M.predict_via_tflite(paths.tflite_path, val_feat)
    yq_test, q_test, ms = M.predict_via_tflite(paths.tflite_path, test_feat)
    assert np.array_equal((yq_test >= 0.5).astype(int), (y_test >= 0.5).astype(int))

    yt = y_test[:, bird:bird + 1]
    yv = y_val[:, bird:bird + 1]
    scores = {
        "keras": (s_val[:, bird:bird + 1], s_test[:, bird:bird + 1], 0.0),
        "tflite": (q_val[:, bird:bird + 1], q_test[:, bird:bird + 1], ms),
    }
    evals: dict[str, dict[str, dict[str, M.EvalResult]]] = {}
    losses: dict[str, dict[str, float]] = {}
    for backend, (sv, sx, ams) in scores.items():
        evals[backend] = {"test": {
            obj: M.eval_result_from_predictions(
                backend=backend, label_names=["bird"],
                y_true=yt, y_score=sx,
                threshold_mode=f"best_{obj}",
                threshold_tuning=(yv, sv),
                avg_inference_ms=ams,
            )
            for obj in ("f1", "f2")
        }}
        losses[backend] = {"test": float(bce(yt, sx).numpy())}

    sidecar = TrainSidecar(
        model=st.spec, collection=DETECTION_COLLECTION,
        batch_size=BATCH_SIZE, seed=SEED, epochs_trained=st.epochs_trained,
    )
    sidecar.save(paths.train_path)
    R.RunRecord.from_evaluations(
        evals, losses=losses,
        tflite_stats=stats[label],
        collection=DETECTION_COLLECTION,
        model_path=str(paths.model_path),
        tflite_path=str(paths.tflite_path),
        train=sidecar,
    ).save(paths.results_path)
    print(f"  [{label}] wrote {paths.results_path}")